# Amazon rivers-on dye metrics (80 days)

Rivers-on only. Computes volume-weighted dye inventories, upper-100-m retention, vertical export, threshold times, and integrated surface-ocean exposure.

In [ ]:
using Oceananigans
using CairoMakie
using Printf

data_directory = raw"C:\Users\meghn\OneDrive\Desktop\summer '26 code\ocean modeling\repo-cleanup\ocean-modeling\amazon_river\new jld2 files\no spinups\80 day salinity dye"
run_prefix = "amazon_rivers_on_validation_FINAL_nz20_gm_off_redi_off_salinity_80day_v2"

dye_file = joinpath(data_directory, "amazon_rivers_on_80d_dye.jld2")
salinity_file = joinpath(data_directory, "amazon_rivers_on_80d_salinity_3d.jld2")

for file in (dye_file, salinity_file)
    @assert isfile(file) "Missing input file: $file. Update data_directory if the 80-day files were moved."
end

dye = FieldTimeSeries(dye_file, "dye"; backend=OnDisk())
salinity = FieldTimeSeries(salinity_file, "S"; backend=OnDisk())

@assert length(dye.times) == length(salinity.times) "Dye and salinity output counts differ"
days_saved = Float64.(dye.times) ./ 86400
println("Loaded $(length(days_saved)) outputs spanning day $(first(days_saved)) to day $(last(days_saved)).")

In [ ]:
# Construct approximate spherical cell volumes. The wet mask removes land and cells below bathymetry.
longitude, latitude, depth = nodes(dye.grid, Center(), Center(), Center())
longitude_faces, latitude_faces, depth_faces = nodes(dye.grid, Face(), Face(), Face())

earth_radius = 6.371e6
delta_longitude = diff(deg2rad.(longitude_faces))
delta_sin_latitude = diff(sin.(deg2rad.(latitude_faces)))
delta_depth = diff(depth_faces)

cell_volume = earth_radius^2 .*
              reshape(delta_longitude, length(delta_longitude), 1, 1) .*
              reshape(delta_sin_latitude, 1, length(delta_sin_latitude), 1) .*
              reshape(delta_depth, 1, 1, length(delta_depth))

initial_salinity = Array(interior(salinity[1]))
wet_cell = isfinite.(initial_salinity) .& (initial_salinity .> 0)
upper_100m = wet_cell .& reshape(depth .>= -100, 1, 1, length(depth))
below_100m = wet_cell .& reshape(depth .< -100, 1, 1, length(depth))

@printf("Active cells: %d; upper-100-m cells: %d\n", count(wet_cell), count(upper_100m))

In [ ]:
total_inventory = zeros(length(days_saved))
upper_100m_inventory = similar(total_inventory)
below_100m_inventory = similar(total_inventory)
minimum_dye = similar(total_inventory)
maximum_dye = similar(total_inventory)

for n in eachindex(days_saved)
    concentration = Float64.(Array(interior(dye[n])))
    concentration[.!isfinite.(concentration)] .= 0
    concentration[.!wet_cell] .= 0

    total_inventory[n] = sum(concentration .* cell_volume)
    upper_100m_inventory[n] = sum(concentration[upper_100m] .* cell_volume[upper_100m])
    below_100m_inventory[n] = sum(concentration[below_100m] .* cell_volume[below_100m])
    minimum_dye[n] = minimum(concentration[wet_cell])
    maximum_dye[n] = maximum(concentration[wet_cell])
end

initial_inventory = total_inventory[1]
domain_fraction = total_inventory ./ initial_inventory
upper_100m_fraction = upper_100m_inventory ./ initial_inventory
below_100m_fraction = below_100m_inventory ./ initial_inventory

# Integral of normalized upper-100-m inventory: effective surface-ocean dye-days.
upper_100m_dye_days = sum(
    0.5 .* (upper_100m_fraction[1:end-1] .+ upper_100m_fraction[2:end]) .* diff(days_saved)
)

function first_threshold_day(values, threshold, days)
    index = findfirst(value -> value <= threshold, values)
    return isnothing(index) ? missing : days[index]
end

t50 = first_threshold_day(upper_100m_fraction, 0.50, days_saved)
tefold = first_threshold_day(upper_100m_fraction, exp(-1), days_saved)
t10 = first_threshold_day(upper_100m_fraction, 0.10, days_saved)

In [ ]:
println("RIVERS-ON 80-DAY DYE SUMMARY")
println("Initial inventory (concentration × m³): ", initial_inventory)
@printf("Final domain fraction:       %.4f\n", domain_fraction[end])
@printf("Final upper-100-m fraction:   %.4f\n", upper_100m_fraction[end])
@printf("Final below-100-m fraction:   %.4f\n", below_100m_fraction[end])
@printf("Upper-100-m exposure:         %.2f dye-days\n", upper_100m_dye_days)
println("Upper-100-m t50:              ", ismissing(t50) ? "> $(last(days_saved)) days" : "$t50 days")
println("Upper-100-m t1/e:             ", ismissing(tefold) ? "> $(last(days_saved)) days" : "$tefold days")
println("Upper-100-m t10:              ", ismissing(t10) ? "> $(last(days_saved)) days" : "$t10 days")
@printf("Dye extrema over run:         %.3e to %.3e\n", minimum(minimum_dye), maximum(maximum_dye))

In [ ]:
figure = Figure(size=(1100, 750))

inventory_axis = Axis(figure[1, 1], xlabel="Simulation day", ylabel="Fraction of initial dye", title="Volume-weighted dye inventory")
lines!(inventory_axis, days_saved, domain_fraction, label="Entire active domain", linewidth=3)
lines!(inventory_axis, days_saved, upper_100m_fraction, label="Upper 100 m", linewidth=3)
lines!(inventory_axis, days_saved, below_100m_fraction, label="Below 100 m", linewidth=3)
axislegend(inventory_axis, position=:rt)

extrema_axis = Axis(figure[2, 1], xlabel="Simulation day", ylabel="Dye concentration", title="Active-ocean dye extrema", yscale=log10)
lines!(extrema_axis, days_saved, max.(maximum_dye, eps()), label="Maximum", linewidth=3)
lines!(extrema_axis, days_saved, max.(abs.(minimum_dye), eps()), label="|Minimum|", linewidth=3)
axislegend(extrema_axis, position=:rt)

figure